# Focus Coach — Phase 2: fine-tune YOLO26n on FPI-Det

Starts from YOLO26n pretrained on COCO (transfer learning) and fine-tunes all layers so it detects two classes: `head` and `phone`.

**How to use:** open this notebook in Colab → Runtime → Change runtime type → **T4 GPU** → edit only the config cell → **Run all**.
If the session disconnects, just **Run all** again: training resumes from the last saved checkpoint on Drive.

> **Note:** the model reported in `results/train/` was trained with an earlier version of this notebook, before the
> data preview, per-class table and before/after cells were added (that run used `EPOCHS = 20`, `batch = 32`).
> This version has not been run end to end on Colab yet. The reporting cells use `ml/train_report.py`, which was
> run locally on the Colab-trained `best.pt`.

**Config.** The only cell you may need to edit: where your repo is, where to keep files on Drive, and the training settings.

In [ ]:
REPO_URL = "https://github.com/majallen3690/focus-coach"  # EDIT: the GitHub repo that holds this project
DRIVE_DIR = "/content/drive/MyDrive/focus-coach"           # everything that must survive a disconnect lives here
EPOCHS = 60
IMGSZ = 640
MODEL = "yolo26n.pt"
PATIENCE = 15
SEED = 42

**GPU check.** Stops with a clear message if no GPU is attached, instead of training for hours on CPU.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again."
DEVICE = 0
print("GPU:", torch.cuda.get_device_name(DEVICE))

**Mount Drive** and create the folder where datasets and runs are kept between sessions.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

**Get the code.** Clones your repo. If it is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left bar → Add new secret → turn on *Notebook access*). The token needs read access to the repo.

In [ ]:
import os
import subprocess

REPO = Path("/content/focus-coach")
assert "YOUR-USERNAME" not in REPO_URL, "Edit REPO_URL in the config cell first."

if not REPO.exists():
    url = REPO_URL
    try:
        from google.colab import userdata

        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:  # no GITHUB_TOKEN secret: assume the repo is public
        pass
    done = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], capture_output=True, text=True)
    if done.returncode:  # print git's message without the token that is inside the URL
        raise RuntimeError(done.stderr.replace(url, REPO_URL))
os.chdir(REPO)
print("repo ready at", REPO)

**Get the data.** If `fpi_yolo.zip` is already on Drive, it is unzipped and nothing is downloaded.
Otherwise the dataset is downloaded from the FPI-Det authors' Google Drive link, cleaned by `ml/prepare_dataset.py`,
and the result is saved to Drive as `fpi_yolo.zip` for later sessions.

> The download branch has not been tried on Colab. If `gdown` fails with a quota message, download the archive in
> your browser from the link in the FPI-Det README, run `ml/prepare_dataset.py` locally, and upload `data/fpi_yolo.zip` to `DRIVE_DIR`.

In [ ]:
import json
import shutil
import sys
import zipfile

DATA_ZIP = Path(DRIVE_DIR) / "fpi_yolo.zip"
DATA_DIR = Path("/content/data/fpi_yolo")
DATA_YAML = DATA_DIR / "fpi.yaml"
FPI_DET_REPO = "https://github.com/KvCgRv/FPI-Det"
FPI_DET_IMAGES = "https://drive.google.com/file/d/1nObrvOSJfj2dzRcW9cGdV6yu6iurWcVo/view?usp=sharing"  # from its README


def run(cmd: list[str]) -> None:
    subprocess.run(cmd, check=True)


if not DATA_ZIP.exists():
    run(["git", "clone", "--depth", "1", FPI_DET_REPO, "third_party/FPI-Det"])
    run(["gdown", "--fuzzy", FPI_DET_IMAGES, "-O", "/content/fpi_det_download"])
    run(["apt-get", "install", "-y", "-qq", "p7zip-full", "p7zip-rar"])  # 7z reads zip and rar alike
    run(["7z", "x", "-y", "/content/fpi_det_download", "-odata/raw"])
    assert Path("data/raw/reorganized_dataset/train/images").exists(), "Unexpected archive layout under data/raw"
    run([sys.executable, "ml/prepare_dataset.py"])
    shutil.copy("data/fpi_yolo.zip", DATA_ZIP)

if not DATA_YAML.exists():
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall("/content/data")

expected = json.loads(Path("results/dataset_stats.json").read_text())
for split in ("train", "val", "test"):
    found = len(list((DATA_DIR / "images" / split).glob("*.jpg")))
    assert found == expected[split]["images"], f"{split}: expected {expected[split]['images']} images, found {found}"
print("dataset ok:", {s: expected[s]["images"] for s in ("train", "val", "test")})

**Install** the pinned packages from `requirements.txt`. If Colab asks you to restart the runtime after this cell, do it and Run all again; nothing is lost.

In [ ]:
%pip install -q -r requirements.txt

**Look at the data.** Eight training images with their boxes, as a sanity check before spending GPU time.

In [ ]:
sys.path.insert(0, str(REPO / "ml"))
import matplotlib.pyplot as plt
import train_report as tr

tr.sample_figure(DATA_DIR / "images" / "train", DATA_DIR / "labels" / "train")
plt.show()

**Train.** Default Ultralytics augmentation is kept (it is recorded in `train_summary.json`). Batch size is picked automatically to fit the GPU.
Checkpoints go to Drive after every epoch. If an interrupted run is found there, it resumes; if a finished run is found, training is skipped.

In [ ]:
import ultralytics
from ultralytics import YOLO

RUNS_DIR = Path(DRIVE_DIR) / "runs"
RUN_NAME = "yolo26n_fpi"
RUN_DIR = RUNS_DIR / RUN_NAME
LAST = RUN_DIR / "weights" / "last.pt"


def run_finished(last: Path) -> bool:
    # Ultralytics sets epoch = -1 when a run ends; an interrupted run keeps its real epoch. Resuming a finished
    # run does not stop: it silently starts a new default training, so it has to be ruled out first.
    return torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) == -1


if LAST.exists() and run_finished(LAST):
    print("Training already finished on Drive: skipping to validation.")
elif LAST.exists():
    print("Interrupted run found: resuming.")
    YOLO(LAST).train(resume=True, data=str(DATA_YAML))
else:
    YOLO(MODEL).train(
        data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=-1, seed=SEED,
        deterministic=True, cache="disk",  # Colab has 2 CPU cores: decoded images on disk avoid a JPEG bottleneck
        project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True,
    )

**Validate** the best checkpoint on the **val** split: precision, recall, mAP50 and mAP50-95 for each class.

In [ ]:
import yaml

BEST = RUN_DIR / "weights" / "best.pt"
run_imgsz = yaml.safe_load((RUN_DIR / "args.yaml").read_text())["imgsz"]
val = tr.per_class_val(BEST, DATA_YAML, DATA_DIR / "images" / "val", run_imgsz, DEVICE)
tr.print_table(val)

**Before vs after.** The COCO-pretrained model and the fine-tuned model on the same 6 val images with small phones, side by side.

In [ ]:
from IPython.display import Image, display

BEFORE_AFTER = RUN_DIR / "before_after.png"
hits = tr.before_after(BEST, MODEL, DATA_DIR / "images" / "val", DATA_DIR / "labels" / "val", BEFORE_AFTER, run_imgsz, DEVICE)
print(f"phone found in {hits['COCO']}/{tr.N_EXAMPLES} images by COCO, {hits['fine-tuned']}/{tr.N_EXAMPLES} by fine-tuned")
display(Image(str(BEFORE_AFTER)))

**Save to Drive.** `best.pt`, `results.csv`, the training plots, the before/after figure, the per-class table and `train_summary.json`
(final metrics, epochs run, training time, GPU, versions, augmentation) go to `DRIVE_DIR/outputs`.

In [ ]:
OUT = Path(DRIVE_DIR) / "outputs"
OUT.mkdir(exist_ok=True)

summary = tr.build_summary(
    RUN_DIR, val, gpu=torch.cuda.get_device_name(DEVICE), torch_version=torch.__version__,
    ultralytics_trained_with=ultralytics.__version__,
    notes=["GPU and library versions are those of the session that wrote this file; a resumed run may have started on another."],
)
(OUT / "train_summary.json").write_text(json.dumps(summary, indent=2))
(OUT / "val_per_class.json").write_text(json.dumps(val, indent=2))

shutil.copy(BEST, OUT / "best.pt")
for name in ("results.csv", "args.yaml", "before_after.png"):
    shutil.copy(RUN_DIR / name, OUT / name)
for plot in RUN_DIR.glob("*.png"):
    shutil.copy(plot, OUT / plot.name)
print("saved to", OUT)
print(sorted(p.name for p in OUT.iterdir()))

## Download and finish

1. In Drive, open `focus-coach/outputs/` (the folder from `DRIVE_DIR`).
2. Download `best.pt` and save it in your repo as `models/best.pt`.
3. Download everything else in that folder and save it in `results/train/`.

Then go back to Claude Code and continue with the next phase.